# PEExcel with Python — Practical How-To

This notebook shows the main workflows for working with a **PEExcel project** through `pexl`.

The examples focus on:

- opening a PEExcel project,
- finding variables and inspecting their metadata,
- selecting variables explicitly or by metadata groups,
- filtering and comparing scenarios,
- creating standard PEExcel diagrams with Plotly,
- building custom pandas / matplotlib plots,
- modifying inputs and exporting a project.

The examples use a PEExcel export workbook with `IN` and `OUT` sheets. Adjust the file path to your own project.

## 0. Setup

During development, `%autoreload` is useful so changes in `pexl` are picked up without restarting the notebook.

For normal use, the two `%autoreload` lines can be omitted.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pexl
from pexl import Project, reporting
from pexl.plot import plotly

## 1. Open a PEExcel project

A workbook is loaded as one `Project`. Each PEExcel scenario / variant column becomes one `Scenario`.

In [ ]:
PROJECT_PATH = Path("../data/exports/ka_project_backup_v1_12.xlsx")

project = Project.from_excel(PROJECT_PATH)
project

In [ ]:
print("Number of scenarios:", len(project))
print("Warnings:", len(project.warnings))
print("Project/group names:", project.project_names())

In [ ]:
# Show import warnings, if there are any
project.warnings[:10]

## 2. Work with scenarios

Scenarios can be accessed by position or by their exact Excel column name.

`column_name` is the structural identifier and is unique within a project. `project_name` and `name` are user-provided semantic labels.

In [ ]:
scenario = project[1]

print("Excel column name:", scenario.column_name)
print("Project/group:", scenario.project_name)
print("Scenario name:", scenario.name)

In [ ]:
column_name = project.column_names()[4]
project[column_name]

In [ ]:
for scenario in project:
    print(
        f"{scenario.column_name:45s}  "
        f"project={scenario.project_name!r}  "
        f"name={scenario.name!r}"
    )

# Finding and selecting variables

The basic workflow is:

1. discover a variable,
2. inspect its definition in the glossary,
3. select it from the project,
4. materialize values only when needed.

## 3. Read known variables

For one scenario, values are available directly through `scenario.v`.

In [ ]:
scenario = project[1]

print("Project description:", scenario.v.project_description)
print("Total GFA:", scenario.v.GFA_total)
print("Office GFA:", scenario.v.GFA_office)

## 4. Use the global PEExcel glossary

Variable definitions are schema-global, so they are exposed through `pexl.glossary`.

Autocomplete is useful here: type `pexl.glossary.` and inspect the available variables.

In [ ]:
meta = pexl.glossary.GFA_total

print("Variable:", meta.var_name)
print("Label:", meta.label_de)
print("Unit:", meta.unit)
print("Source:", meta.source)
print("Domain:", meta.domain)
print("Measure:", meta.measure)

The glossary gives you the metadata object without needing a particular scenario.

In [ ]:
pexl.glossary.PEI_virtual_demand

## 5. Discover variables

`project.variables` is the selectable collection of schema variables.

Use it when you do not yet know the exact variable name.

In [ ]:
print("Number of schema variables:", len(project.variables))

print("\nFirst variables:")
print(*project.variables.var_names[:25], sep="\n")

A simple text search is often enough for exploratory work:

In [ ]:
[name for name in project.variables.var_names if "PEI" in name][:30]

Input and output views expose their own variable selections:

In [ ]:
print("First OUT variables:")
print(*project.out.variables.var_names[:20], sep="\n")

## 6. Select explicit variables

You can select variables by canonical name:

In [ ]:
view = project.view.select(
    "PEI_virtual_demand",
    "PEI_virtual_supply",
)

view

Or use the discovered glossary objects directly:

In [ ]:
view = project.view.select(
    pexl.glossary.PEI_virtual_demand,
    pexl.glossary.PEI_virtual_supply,
)

view

Materialize a selection as a pandas DataFrame only when you need tabular analysis or plotting:

In [ ]:
df = view.to_frame()
df.head()

# Metadata groups

Once you know the basic variable workflow, metadata groups are useful for selecting larger families of related variables.

## 7. Discover domains, measures and entity groups

In [ ]:
print("OUT domains:", end="\n* ")
print(*project.out.domains(), sep="\n* ")

print("\nOUT measures:", end="\n* ")
print(*project.out.measures(), sep="\n* ")

In [ ]:
print("IN domains:", end="\n* ")
print(*project.inn.domains(), sep="\n* ")

print("\nIN measures:", end="\n* ")
print(*project.inn.measures(), sep="\n* ")

print("\nIN entity groups:", end="\n* ")
print(*project.inn.entity_groups(), sep="\n* ")

## 8. Select variables by metadata

For example, select the complete primary-energy balance:

In [ ]:
pe = project.out.select(
    domain="primary_energy_balance"
)

pe

Metadata filters can be combined:

In [ ]:
pe_demand = project.out.select(
    domain="primary_energy_balance",
    measure="demand",
)

print("Shape:", pe_demand.shape)
print("Variables:")
print(*pe_demand.variables.var_names, sep="\n")

The variable and scenario selections behave like read-only collections:

In [ ]:
print("First scenario:", pe_demand.scenarios[0])
print("First variable:", pe_demand.variables[0])

for meta in pe_demand.variables[:3]:
    print(meta.var_name, meta.unit)

## 9. Filter scenarios

Use `.where(...)` to restrict the scenario axis.

In [ ]:
forsthaus = project.where(
    project_name="Forsthausgasse"
)

for scenario in forsthaus.scenarios:
    print(scenario.column_name)

Scenario and variable filters can be chained in either order:

In [ ]:
comparison = (
    project.out
    .where(project_name="Forsthausgasse")
    .select(domain="primary_energy_balance")
)

comparison

In [ ]:
a = (
    project.out
    .select(domain="primary_energy_balance")
    .where(project_name="Forsthausgasse")
)

b = (
    project.out
    .where(project_name="Forsthausgasse")
    .select(domain="primary_energy_balance")
)

print(a.shape, b.shape)

To inspect the values available for one scenario attribute or variable, use `unique(...)` on the scenario selection:

In [ ]:
project.view.scenarios.unique("project_name")

In [ ]:
project.view.scenarios.unique(
    "preset_recorded_heating_system"
)

## 10. Convert selections to DataFrames and compare scenarios

In [ ]:
df = comparison.to_frame()
df.head()

For a difference relative to the first scenario:

In [ ]:
reference = df.iloc[0]

difference = df.subtract(reference)
difference.head()

For relative differences in percent:

In [ ]:
relative_difference = (
    df.subtract(reference)
      .divide(reference)
      .multiply(100)
)

relative_difference.head()

# Standard PEExcel diagrams

Standard diagrams use the report definitions generated from the PEExcel schema. The report layer determines the variables, roles, labels, colors and ordering; Plotly only renders the result.

## 11. Discover available standard diagrams

In [ ]:
reporting.chart_names()

## 12. Primary-energy balance

In [ ]:
report = reporting.materialize(
    project.view,
    "primary_energy_balance",
)

fig = plotly.render(report)
fig.show()

The same report can be materialized for a filtered scenario view:

In [ ]:
view = project.where(
    project_name="Forsthausgasse"
)

report = reporting.materialize(
    view,
    "primary_energy_balance",
)

plotly.render(report).show()

## 13. Heat balance

In [ ]:
report = reporting.materialize(
    project.view,
    "heat_balance",
)

plotly.render(report).show()

## 14. Final-energy balance

In [ ]:
report = reporting.materialize(
    project.view,
    "final_energy_balance",
)

plotly.render(report).show()

The materialized report also exposes its underlying DataFrame for further analysis:

In [ ]:
report["frame"].head()

# Custom analysis and plotting

For analyses that are not standard PEExcel reports, select the required variables and use normal pandas, matplotlib or Plotly workflows.

## 15. Custom primary-energy demand / supply scatter plot

This example uses the discovered `PEI_virtual_demand` and `PEI_virtual_supply` variables.

The plot keeps the formatting of the original analysis: equal axes, a dotted 1:1 balance line, restrained scenario markers and publication-style labels.

In [ ]:
df = project.view.select(
    pexl.glossary.PEI_virtual_demand,
    pexl.glossary.PEI_virtual_supply,
).to_frame()

x = pd.to_numeric(df["PEI_virtual_demand"], errors="coerce")
y = pd.to_numeric(df["PEI_virtual_supply"], errors="coerce")

limit = np.nanmax([x.max(), y.max()])
limit = max(50, np.ceil(limit / 50) * 50)

fig, ax = plt.subplots(1, 1, figsize=(7, 4.3))

ax.plot(
    [0, limit],
    [0, limit],
    color="green",
    linestyle="dotted",
    linewidth=0.8,
    label="Demand = supply",
)

ax.scatter(
    x,
    y,
    color="darkgrey",
    label="Scenarios",
)

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)

ticks = np.arange(0, limit + 1, 50)
ax.set_xticks(ticks)
ax.set_yticks(ticks)

ax.set_xlabel(
    "Primary Energy Demand [kWh/m²NFA]",
    fontname="Palatino Linotype",
)
ax.set_ylabel(
    "Primary Energy Supply [kWh/m²NFA]",
    fontname="Palatino Linotype",
)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

Because `df` is an ordinary DataFrame, scenario families, parameter sweeps or other analysis-specific groupings can be added as columns and passed directly to matplotlib, seaborn or Plotly.

# One-scenario workflows

## 16. Select results from one scenario

In [ ]:
scenario = project[1]

heat = scenario.out.select(
    domain="heat_balance"
)

heat

In [ ]:
heat.to_var_dict()

In [ ]:
print("QH:", heat.QH)

## 17. Select related inputs

Example: residential-use inputs:

In [ ]:
residential_inputs = scenario.inn.select(
    entity_group="usage",
    entity_key="residential",
)

residential_inputs.to_var_dict()

Example: heating-system inputs:

In [ ]:
heating_inputs = scenario.inn.select(
    entity_group="heating"
)

heating_inputs

## 18. Change input values

Scenario values are changed through `scenario.v`.

The actual assignment is commented out so running the notebook does not modify the project accidentally.

In [ ]:
scenario_edit = project[1]

print("Before:", scenario_edit.v.GFA_office)

# scenario_edit.v.GFA_office = 250.0

print("After:", scenario_edit.v.GFA_office)

PEExcel formulas are not recalculated by Python automatically unless the workflow explicitly runs Excel or another calculation step.

# Creating and exporting projects

## 19. Create a project or scenario

In [ ]:
from pexl.model.scenario import Scenario

new_project = Project()

new_scenario = Scenario(
    "Example Project | Variant A"
)

new_scenario.v.project_name = "Example Project"
new_scenario.v.project_scenario_name = "Variant A"
new_scenario.v.GFA_office = 250.0

new_project.add_scenario(new_scenario)

new_project

When creating a new scenario, use a unique `column_name`. The conventional form is:

```text
project_name | project_scenario_name
```

## 20. Export a project

In [ ]:
OUTPUT_PATH = Path("../data/exports/new_project.xlsx")

# Uncomment to write the file:
# new_project.to_excel(OUTPUT_PATH)

# Quick reference

| Task | Example |
|---|---|
| Open project | `project = Project.from_excel(path)` |
| First scenario | `project[0]` |
| Scenario by exact column | `project["Project | Variant"]` |
| Read one value | `scenario.v.GFA_total` |
| Discover metadata | `pexl.glossary.GFA_total` |
| List variables | `project.variables.var_names` |
| Select explicit variables | `project.view.select("A", "B")` |
| Select glossary variables | `project.view.select(pexl.glossary.A, pexl.glossary.B)` |
| Select inputs | `project.inn.select(...)` |
| Select outputs | `project.out.select(...)` |
| Select by domain | `project.out.select(domain="...")` |
| Filter scenarios | `project.where(project_name="...")` |
| List domains | `project.out.domains()` |
| List measures | `project.out.measures()` |
| Unique scenario values | `project.view.scenarios.unique("project_name")` |
| DataFrame | `view.to_frame()` |
| Available standard charts | `reporting.chart_names()` |
| Standard Plotly chart | `plotly.render(reporting.materialize(project.view, "heat_balance"))` |
| Modify value | `scenario.v.GFA_office = 250.0` |
| Export | `project.to_excel(path)` |